<img src="../img/GTK_Logo_Social_Icon.jpg" width=175 align="right" />


# Worksheet 9.4: Tokenizers & Embeddings — Answers

*Module 9 — Build Your Own LLM.* This is the **answer key** with every cell completed.

A neural network only understands **numbers**, but language is made of **text**. The first job in any LLM is **tokenization**: chopping text into pieces (*tokens*) and mapping each to an integer id.

Real LLMs — GPT, Llama, Claude — all use **BPE** (Byte-Pair Encoding). In this lab you'll *train your own BPE tokenizer* on the course corpus, look at what it learned, and then use it to build your first language model: a **bigram** model. It won't write essays, but you'll watch pure noise turn into word-shaped text.

## 1. Load the text

We use two corpora: a chunk of Shakespeare (general English) and ~1,300 real **CVE vulnerability descriptions**. We train the tokenizer on **both together** so that a single vocabulary covers general English *and* security language — we'll need that in Lab 9.6.

In [1]:
general = open("../data/tiny_corpus.txt").read()     # general English (Shakespeare)
cyber   = open("../data/cyber_corpus.txt").read()    # real CVE descriptions
corpus  = general + "\n" + cyber

print("general chars:", len(general), "| cyber chars:", len(cyber))
print("distinct characters:", len(set(corpus)))
print("\n--- Shakespeare ---\n", general[:180])
print("\n--- CVE ---\n", cyber[:180])

general chars: 319124 | cyber chars: 319998
distinct characters: 93

--- Shakespeare ---
 First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First

--- CVE ---
 CVE-2024-21732: FlyCms through abbaa5a allows XSS via the permission management feature.
CVE-2023-5877: The affiliate-toolkit WordPress plugin before 3.4.3 lacks authorization and 


## 2. Start from bytes

BPE starts below the level of characters: **raw UTF-8 bytes**. That gives us a fixed starting vocabulary of exactly 256 ids and — importantly — means *no text is ever unrepresentable*. There is no `<unknown>` token, because every possible input is just bytes.

In [2]:
ids = list(corpus.encode("utf-8"))
print("corpus as bytes:", len(ids), "tokens, vocabulary of 256")
print("first 20 byte ids:", ids[:20])
print("decoded back     :", repr(bytes(ids[:20]).decode("utf-8")))

corpus as bytes: 639123 tokens, vocabulary of 256
first 20 byte ids: [70, 105, 114, 115, 116, 32, 67, 105, 116, 105, 122, 101, 110, 58, 10, 66, 101, 102, 111, 114]
decoded back     : 'First Citizen:\nBefor'


## 3. The BPE algorithm: merge the most frequent pair

The whole algorithm is one sentence: **find the most frequent adjacent pair of tokens, and replace it everywhere with a single new token.** Repeat.

Each merge adds one entry to the vocabulary. That's it — no linguistics, no word lists, no rules about what a "word" is.

**TODO:** find the most common adjacent pair, then replace every occurrence of it with `new_id`.

In [3]:
import collections

def most_common_pair(ids):
    return collections.Counter(zip(ids, ids[1:])).most_common(1)[0]

def merge(ids, pair, new_id):
    """Replace every adjacent occurrence of `pair` with the single token `new_id`."""
    out, i, n = [], 0, len(ids)
    while i < n:
        if i < n - 1 and ids[i] == pair[0] and ids[i + 1] == pair[1]:
            out.append(new_id); i += 2
        else:
            out.append(ids[i]); i += 1
    return out

# Watch three merges happen on a toy string first.
demo = list("low lower lowest newest widest".encode("utf-8"))
demo_vocab = {i: bytes([i]) for i in range(256)}
for k in range(3):
    pair, count = most_common_pair(demo)
    new_id = 256 + k
    demo_vocab[new_id] = demo_vocab[pair[0]] + demo_vocab[pair[1]]
    demo = merge(demo, pair, new_id)
    print(f"merge {k}: {demo_vocab[pair[0]]!r} + {demo_vocab[pair[1]]!r} "
          f"(seen {count}x) -> {demo_vocab[new_id]!r}")

merge 0: b'l' + b'o' (seen 3x) -> b'lo'
merge 1: b'lo' + b'w' (seen 3x) -> b'low'
merge 2: b'e' + b's' (seen 3x) -> b'es'


## 4. Train the real tokenizer

Now the same loop on the full corpus, for **512 merges** — giving a vocabulary of `256 + 512 = 768`.

This takes about 20–30 seconds. (Real tokenizers do 50,000+ merges with a much faster implementation; the algorithm is identical.)

In [4]:
NUM_MERGES = 512

ids = list(corpus.encode("utf-8"))
merges = {}                                    # (a, b) -> new_id, in training order
vocab  = {i: bytes([i]) for i in range(256)}   # id -> the bytes it stands for

for k in range(NUM_MERGES):
    pair, count = most_common_pair(ids)
    if count < 2:
        break
    new_id = 256 + k
    merges[pair] = new_id
    vocab[new_id] = vocab[pair[0]] + vocab[pair[1]]
    ids = merge(ids, pair, new_id)

vocab_size = 256 + len(merges)
print("vocab size:", vocab_size)
print("corpus went from", len(corpus.encode("utf-8")), "byte tokens to", len(ids), "BPE tokens")
print(f"compression: {len(corpus) / len(ids):.2f} characters per token")

vocab size: 768
corpus went from 639123 byte tokens to 273050 BPE tokens
compression: 2.34 characters per token


## 5. What did it learn?

This is the interesting part. Nobody told the tokenizer about English, about security, or even about words. Look at the longest tokens it invented:

In [5]:
longest = sorted(vocab.values(), key=len, reverse=True)[:24]
for t in longest:
    print(repr(t.decode("utf-8", errors="replace")))

'vulnerability '
'authenticated '
'vulnerabiliti'
'vulnerabilit'
'.\nCVE-2023-5'
'.\nCVE-2023-4'
'.\nCVE-2023-3'
'GLOUCESTER:\n'
'CORIOLANUS:\n'
'.\nCVE-2023-'
'arbitrary '
'.\nCVE-2024'
'MENENIUS:\n'
'.\nCVE-202'
'has been '
'authentic'
'versions '
'possible '
'vulnerab'
' to the '
'attacker'
'crafted '
'function'
'lead to '


Purely by counting bytes, it discovered `attacker`, `vulnerability `, `arbitrary `, and the `CVE-2023-` prefix — alongside Shakespeare's speaker tags like `GLOUCESTER:\n`. **The security vocabulary of your corpus is now literally built into the model's alphabet.**

This matters for more than compression. Tokenizer boundaries are where a surprising amount of LLM weirdness lives — "glitch tokens", strange behaviour on rare strings, and some prompt-injection tricks all trace back to how text got chopped up here.

## 6. Encode and decode

To encode new text, apply the learned merges **in the order they were learned**. To decode, look each id up and concatenate the bytes.

Note the `errors="replace"` in `decode`: a model generating tokens freely can emit a *partial* multi-byte character, and we'd rather print `\ufffd` than crash.

**TODO:** finish `encode` by applying each merge in turn.

In [6]:
def encode(s):
    ids = list(s.encode("utf-8"))
    for pair, new_id in merges.items():      # dicts keep insertion order = training order
        ids = merge(ids, pair, new_id)
    return ids

def decode(ids):
    return b"".join(vocab[i] for i in ids).decode("utf-8", errors="replace")

for s in ["CVE-2023-1234: the attacker can execute arbitrary code.", "To be, or not to be"]:
    enc = encode(s)
    print(f"{len(s):3d} chars -> {len(enc):3d} tokens | round-trips: {decode(enc) == s}")
    print("   ", [decode([i]) for i in enc][:14], "...\n")

 55 chars ->  17 tokens | round-trips: True
    ['CVE-202', '3-', '1', '2', '3', '4', ':', ' the ', 'attack', 'er ', 'can ', 'execut', 'e ', 'arbitrary '] ...

 19 chars ->   7 tokens | round-trips: True
    ['To ', 'be', ', ', 'or ', 'not', ' to ', 'be'] ...



## 7. Save it for the other labs

Labs 9.5 and 9.6 load this same tokenizer instead of retraining it, so token id 42 means the same thing everywhere.

In [7]:
import json
json.dump([[a, b, i] for (a, b), i in merges.items()], open("../data/bpe_merges.json", "w"))
print("saved", len(merges), "merges to ../data/bpe_merges.json")

import torch
data = torch.tensor(encode(general), dtype=torch.long)   # we model general English first
print("encoded Shakespeare:", data.shape[0], "tokens")

saved 512 merges to ../data/bpe_merges.json
encoded Shakespeare: 145098 tokens


## 8. Embeddings: ids become vectors

An id like `42` is just a label — it carries no meaning. An **embedding table** is a lookup table where row `i` holds a vector of learnable numbers for token `i`. During training those vectors move so that tokens used in similar ways end up with similar vectors.

In [8]:
import torch.nn as nn
import torch.nn.functional as F

device = ("cuda" if torch.cuda.is_available()
          else "mps" if torch.backends.mps.is_available() else "cpu")
torch.manual_seed(1337)

emb = nn.Embedding(num_embeddings=vocab_size, embedding_dim=8)
sample_ids = torch.tensor(encode("attacker"))
print("tokens:", [decode([i]) for i in sample_ids.tolist()])
print("their embedding vectors:\n", emb(sample_ids))
print("table shape (vocab_size x dim):", emb.weight.shape)
print("device:", device)

tokens: ['attacker']
their embedding vectors:
 tensor([[-0.9975,  0.1778, -0.3417, -1.0127, -0.2784, -1.4193,  0.6370, -2.0427]],
       grad_fn=<EmbeddingBackward0>)
table shape (vocab_size x dim): torch.Size([768, 8])
device: mps


## 9. Make training batches

The model learns by example: given a chunk of text, predict the **next** token at every position. So the targets are simply the inputs **shifted by one**.

In [9]:
n = int(0.9 * len(data))
train_data, val_data = data[:n], data[n:]

block_size = 8    # tokens of context
batch_size = 32   # chunks per step

def get_batch(split):
    d = train_data if split == "train" else val_data
    ix = torch.randint(len(d) - block_size, (batch_size,))
    x = torch.stack([d[i:i + block_size] for i in ix])
    y = torch.stack([d[i + 1:i + block_size + 1] for i in ix])
    return x.to(device), y.to(device)

xb, yb = get_batch("train")
print("inputs  shape:", xb.shape)
print("targets shape:", yb.shape, "(inputs shifted by one)")
print("\none input chunk decoded:", repr(decode(xb[0].tolist())))

inputs  shape: torch.Size([32, 8])
targets shape: torch.Size([32, 8]) (inputs shifted by one)

one input chunk decoded: "like to do't as an"


## 10. The bigram model

The idea: each token looks up a row that *is* the scores (**logits**) for what comes next. We measure error with **cross-entropy** (how surprised the model is by the true next token) and **generate** by repeatedly sampling the next token and feeding it back in.

**TODO:** finish the loss in `forward` and the sampling step in `generate`.

In [10]:
class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        # row i = logits for the token that follows token i
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):
        logits = self.token_embedding_table(idx)        # (B, T, vocab_size)
        loss = None
        if targets is not None:
            B, T, C = logits.shape
            loss = F.cross_entropy(logits.view(B * T, C), targets.view(B * T))
        return logits, loss

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            logits, _ = self(idx)               # (B, T, vocab_size)
            logits = logits[:, -1, :]           # only the last position matters
            probs = F.softmax(logits, dim=-1)   # scores -> probabilities
            next_id = torch.multinomial(probs, num_samples=1)
            idx = torch.cat([idx, next_id], dim=1)
        return idx

model = BigramLanguageModel(vocab_size).to(device)
print("model ready with", sum(p.numel() for p in model.parameters()), "parameters")

model ready with 589824 parameters


## 11. Generate BEFORE training

With random weights the model produces gibberish. We seed generation with a newline token.

A useful number to know: an untrained model over a vocabulary of 768 should have a loss near `ln(768) ≈ 6.64` — that's what "no idea whatsoever" looks like. Anything below that means it has learned *something*.

In [11]:
start = torch.tensor([encode("\n")], dtype=torch.long, device=device)
print("expected loss of an untrained model:", round(torch.tensor(float(vocab_size)).log().item(), 2))
print("\nBEFORE training (random gibberish):\n")
print(decode(model.generate(start, max_new_tokens=100)[0].tolist()))

expected loss of an untrained model: 6.64

BEFORE training (random gibberish):


This alloirst so se  thy not ) andCom. This comallows . A BA you(ress �attackain ated LOattpuces,
trlead to 'dak�good , and RICH�2good of ��lead to �usondisc+�ET~�ome idhowThe lead to ll agarZress disclo��
�e th.

agha, and . The : A A�;
pher kƚ�asover : A �remoper��authenticated 


## 12. Train

The same loop as Lab 9.3, with PyTorch's `AdamW` optimizer doing the updates. Watch the loss fall away from 6.64.

In [12]:
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-2)
for step in range(3000):
    xb, yb = get_batch("train")
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
    if step % 500 == 0:
        print(f"step {step:4d} | loss {loss.item():.3f}")
print("final loss:", round(loss.item(), 3))

step    0 | loss 7.098
step  500 | loss 4.747
step 1000 | loss 4.047
step 1500 | loss 3.904
step 2000 | loss 3.573
step 2500 | loss 3.553
final loss: 3.737


## 13. Generate AFTER training

Because our tokens are word *pieces*, the output already contains real words — but strung together incoherently. A bigram only ever looks at **one** previous token, so it cannot track context. Fixing that — letting tokens look further back — is **attention**, the subject of Lab 9.5.

In [13]:
print("AFTER training (real words, no coherence):\n")
print(decode(model.generate(start, max_new_tokens=200)[0].tolist()))

AFTER training (real words, no coherence):


Thy lordship!

MENENIUS:
And cantom might,
Thy't, best nee's, let wondon afforthy s;
Andome, comethephesby the heavend Clysmocrothatry.

COJupon depose: the e.
When damnly he cantwould may, you you ys,
That reast is bly of overes I nor scor:
Anilt dome, s. Your intent-dche; and the who ful E:
You on annot in the worlack:
What would not
poreaillampates, up
Hastinger,
To ft in so ses,
Oueldid	youre more res wo!

CORIOLANUS:
Come, sy


## Recap

- **BPE tokenization** starts from raw bytes and repeatedly merges the most frequent adjacent pair. You trained one; it learned `attacker`, `vulnerability `, and `CVE-2023-` on its own.
- Because it starts from bytes, there is **no `<unknown>` token** — any input is representable.
- **Embeddings** turn token ids into learnable vectors.
- We framed language modeling as **predict the next token**, scored with **cross-entropy**, and trained a **bigram** model.
- Its weakness — seeing only one token of context — motivates **attention**.

Next: **Worksheet 9.5 — Attention & Your First GPT**.